# Projekt 02 (medium): Einen schmutzigen Datensatz retten

**Ziel:** Ein realistisch verschmutzter Bestell-Datensatz (500 Bestellungen eines
Online-Shops) soll analysierbar gemacht werden. Du findest die Probleme **selbst**,
behandelst sie **begruendet** und weist am Ende mit einem Abnahmetest nach, dass
alles sauber ist.

**Vorbereitung** (einmalig, im Ordner `02-medium`, venv aktiv):

```
python generate_data.py
```

Die Daten sind **synthetisch** (warum, steht im Kopf von `generate_data.py`) —
aber jedes eingebaute Problem kommt so in echten Daten staendig vor.
**Nicht schummeln:** Erst selbst suchen, dann ggf. im Generator nachsehen.

**Bezug zum Skript:** Abschnitte 2.1 (Bereinigung), 2.3 (groupby), 1.4 (Boxplot).

**Arbeitsregel fuer dieses Projekt:** Zu jeder Bereinigungsentscheidung schreibst du
einen Satz Begruendung in die dafuer vorgesehenen Markdown-Zellen („Entscheidung: …").
Das ist keine Schikane — undokumentierte Bereinigung ist nicht reproduzierbar (Skript 3.3).

## 1. Sichten: Was stimmt hier alles nicht?

**Aufgabe:** Verschaffe dir mit `head()`, `info()`, `describe(include="all")` und
Stichproben einen Ueberblick und notiere in der Markdown-Zelle darunter eine Liste
aller Probleme, die du entdeckst. (Es sind mindestens 6.)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

roh = pd.read_csv("daten/bestellungen_roh.csv") if __import__("os").path.exists("daten/bestellungen_roh.csv") \
      else pd.read_csv("../daten/bestellungen_roh.csv")
print(roh.shape)
roh.head(8)

In [ ]:
roh.info()
roh.describe(include="all").T

**Deine Problemliste** *(hier eintragen, bevor du weiterliest!)*:

1. …
2. …

<details><summary>Vergleich: die komplette Liste (erst nach eigener Suche aufklappen)</summary>

1. `preis` ist Text („49,99 EUR") statt Zahl — Komma & Einheit
2. fehlende Preise (leere Felder → NaN)
3. extreme Preis-Ausreisser (vierstellig+ — sieht nach Kommafehler aus)
4. `kunden_alter` enthaelt den Sondercode −999 (und mind. einen unmoeglichen Wert)
5. `stadt` inkonsistent: Gross-/Kleinschreibung, Leerzeichen, Umlaut-Varianten
6. `datum` in zwei Formaten gemischt (ISO und deutsch)
7. Duplikat-Zeilen (515 Zeilen, aber nur 500 verschiedene `bestell_id`s?)
</details>

## 2. Duplikate

**Aufgabe:** Wie viele exakte Duplikate gibt es? Entferne sie (Ergebnis: `df`).
Pruefe danach, ob auch `bestell_id` eindeutig ist (sonst gaebe es Fast-Duplikate).

In [ ]:
# TODO: roh.duplicated().sum(), drop_duplicates(), df["bestell_id"].is_unique
print("Zeilen:", len(df))   # erwartet: 500

**Entscheidung:** Exakte Duplikate entfernt (15 Stueck) — identische Zeilen inkl.
Bestell-ID koennen keine echten Zweitbestellungen sein, sondern sind ein Exportfehler.

## 3. Preis: Text → Zahl, fehlende Werte

**Aufgabe:** Erzeuge eine numerische Spalte `preis_eur` (Einheit entfernen, Komma →
Punkt, `astype(float)` — leere Felder sollen automatisch `NaN` werden).
Wie viele Preise fehlen?

In [ ]:
# TODO: df["preis_eur"] = ... (str.replace zweimal, dann astype(float))
print("fehlende Preise:", df["preis_eur"].isna().sum())   # erwartet: 12

**Entscheidung:** Die 12 fehlenden Preise bleiben `NaN` (kein Imputieren): Es sind
nur 2,4 % der Zeilen, pandas-Aggregationen ignorieren NaN automatisch, und einen
Bestellpreis mit dem Median aufzufuellen wuerde eine Genauigkeit vortaeuschen,
die wir nicht haben.

## 4. Ausreisser — die Lieblingsfalle

**Aufgabe (a):** Wende die IQR-Regel (Skript 2.1) auf `preis_eur` **global** an.
Wie viele Zeilen werden markiert? Schau dir die markierten Preise an — sind das
alles Fehler?

In [ ]:
# TODO: q1, q3 = df["preis_eur"].quantile([0.25, 0.75]); iqr; obere Grenze; Maske; Anzahl + Werte ausgeben

**41 markierte Zeilen — aber nur 3 sehen wirklich kaputt aus** (vierstellig+).
Der Rest sind teure, aber plausible Elektronik-Preise. Die globale IQR-Regel
vergleicht Buecher mit Fernsehern — unfair. Der Boxplot pro Kategorie zeigt das Problem:

In [ ]:
import seaborn as sns
sns.boxplot(data=df, x="kategorie", y="preis_eur")
plt.yscale("log")   # log-Skala, sonst quetschen die Ausreisser alles zusammen
plt.title("Preise pro Kategorie (log-Skala) — die 3 echten Ausreisser stechen heraus")
plt.show()

**Aufgabe (b):** Wende die IQR-Regel **pro Kategorie** an
(Tipp: `df.groupby("kategorie")["preis_eur"].transform(...)` mit einer Funktion,
die je Gruppe die obere Grenze liefert). Jetzt sollten genau **3** Zeilen markiert
werden. Korrigiere sie: Es sind offensichtlich Kommafehler (Faktor 100) → teile durch 100.

In [ ]:
# TODO: Funktion obere_iqr_grenze(g); transform; Maske; markierte Zeilen anzeigen; /100 korrigieren
print("neuer Maximalpreis:", df["preis_eur"].max().round(2))

**Entscheidung:** 3 Preise (1321 / 18363 / 19685 EUR) als Kommafehler eingestuft und
durch 100 geteilt — Begruendung: Sie liegen um genau zwei Groessenordnungen ueber dem
Kategorie-Ueblichen, und nach der Korrektur passen sie unauffaellig ins Preisband.
*(Im echten Leben wuerde man zusaetzlich im Quellsystem nachpruefen.)*
Beachte den Dreiklang aus Skript 2.1: Fehler → korrigieren; echter Extremwert →
behalten; Sondercode → NaN. Hier lagen Fehler vor — die teuren Fernseher von
oben waren dagegen echte Extremwerte und bleiben unangetastet!

## 5. Staedte normalisieren

**Aufgabe:** `df["stadt"].value_counts()` zeigt das Chaos. Erzeuge `stadt_sauber`:
Leerzeichen weg, Kleinschreibung, Umlautvarianten vereinheitlichen
(`münchen`→`muenchen`, `köln`→`koeln`), dann `str.capitalize()`.
Am Ende: genau **5** verschiedene Staedte.

In [ ]:
print(df["stadt"].value_counts())
# TODO: df["stadt_sauber"] = ... (strip, lower, replace-Mapping, capitalize)
print("\nnachher:", sorted(df["stadt_sauber"].unique()))
print("Anzahl:", df["stadt_sauber"].nunique())   # erwartet: 5

## 6. Alter: Sondercode und Plausibilitaet

**Aufgabe:** `df["kunden_alter"].describe()` verraet zwei Probleme (min und max
anschauen!). Erzeuge `alter`: den Sondercode **−999** („keine Angabe") durch `NaN`
ersetzen und unplausible Alter (> 100) ebenfalls auf `NaN` setzen.

In [ ]:
print(df["kunden_alter"].describe().round(1))
# TODO: df["alter"] = ... (replace(-999, np.nan); dann alter > 100 auf np.nan)
print("\nfehlend:", df["alter"].isna().sum(), "| Bereich:", df["alter"].min(), "-", df["alter"].max())

**Entscheidung:** −999 ist offensichtlich ein „keine Angabe"-Code (25×) → NaN.
Das Alter 234 ist physisch unmoeglich, der wahre Wert unrekonstruierbar → NaN
statt raten. Insgesamt 26 fehlende Alter (5,2 %) — fuer Analysen nach Alter
werden diese Zeilen automatisch ausgelassen, der Rest des Datensatzes bleibt nutzbar.

## 7. Datum: zwei Formate, eine Spalte

**Aufgabe:** `pd.to_datetime` mit festem `format` und `errors="coerce"` parst nur
das passende Format (Rest wird NaT). Parse beide Formate getrennt und fuege sie mit
`fillna` zusammen. Am Ende: **0** NaT-Werte.

In [ ]:
# TODO: beide Formate einzeln parsen (errors="coerce"), mit fillna kombinieren
print("nicht parsebar:", df["datum_sauber"].isna().sum())   # erwartet: 0
print(df["datum_sauber"].min(), "bis", df["datum_sauber"].max())

## 8. Abnahmetest

Diese Zelle ist fertig vorgegeben. Laeuft sie ohne Fehler durch, ist dein Datensatz
offiziell sauber.

In [ ]:
sauber = df[["bestell_id", "datum_sauber", "stadt_sauber", "kategorie",
             "preis_eur", "menge", "alter"]].rename(
                 columns={"datum_sauber": "datum", "stadt_sauber": "stadt"})

assert len(sauber) == 500, "Duplikate nicht (richtig) entfernt"
assert sauber["bestell_id"].is_unique
assert sauber["preis_eur"].dtype == float and sauber["preis_eur"].isna().sum() == 12
assert sauber["preis_eur"].max() < 500, "Preis-Ausreisser nicht korrigiert"
assert sauber["stadt"].nunique() == 5, "Staedte nicht vollstaendig normalisiert"
assert sauber["alter"].isna().sum() == 26 and sauber["alter"].max() <= 100
assert str(sauber["datum"].dtype).startswith("datetime64") and sauber["datum"].isna().sum() == 0
assert abs(sauber["preis_eur"].median() - 71.6) < 0.5, "Preise stimmen nicht (Korrektur pruefen)"

print("ABNAHME BESTANDEN — Datensatz ist sauber. 🎉")
sauber.head()

## 9. Belohnung: die erste echte Analyse

Jetzt, wo die Daten sauber sind, ist die Analyse ein Dreizeiler — fertig vorgegeben.

In [ ]:
fig, achsen = plt.subplots(1, 2, figsize=(12, 4))
sauber.groupby("kategorie")["preis_eur"].mean().sort_values().plot.barh(ax=achsen[0])
achsen[0].set_title("Mittlerer Preis pro Kategorie (EUR)")
sauber.set_index("datum").resample("ME")["preis_eur"].sum().plot(ax=achsen[1])
achsen[1].set_title("Umsatz pro Monat (EUR)")
plt.tight_layout(); plt.show()

sauber.groupby("stadt").agg(bestellungen=("bestell_id", "count"),
                            umsatz=("preis_eur", "sum"),
                            mittleres_alter=("alter", "mean")).round(1)

## Geschafft — was du jetzt kannst

- Datenprobleme systematisch *finden* statt nur beheben
- Typkonvertierung, Duplikate, Sondercodes, Textnormalisierung, gemischte Datumsformate
- die IQR-Regel richtig einsetzen — inklusive der Lektion, dass sie **pro Gruppe**
  angewendet werden muss, wenn die Gruppen verschiedene Skalen haben
- Bereinigungsentscheidungen dokumentieren und per Abnahmetest absichern

**Bonusaufgaben** (optional):
1. Oeffne `generate_data.py` und vergleiche: Hast du alle eingebauten Probleme gefunden?
2. Aendere im Generator den Seed — laeuft dein Notebook trotzdem durch? (Die
   Abnahme-Zahlen aendern sich; welche Pruefungen sind seed-unabhaengig formulierbar?)
3. Baue eine Funktion `bereinige(df_roh)`, die alle Schritte kapselt — der erste
   Schritt von der Analyse zur wiederverwendbaren Pipeline.